In [1]:
from pyathena import connect
from dotenv import load_dotenv
import os
import pandas as pd
from datetime import date

load_dotenv()


CONN = connect(
    aws_access_key_id=os.getenv("AWS_API_ID"),
    aws_secret_access_key=os.getenv("AWS_API_KEY"),
    s3_staging_dir=os.getenv("AWS_S3"),
    region_name=os.getenv("AWS_REGION")
)

In [60]:
# load_ids = {
#     'cierre_base': '691450ebe0e4bd2690be54f0',
#     'cierre_up':'69145da2e0e4bd2690c64b39',
#     'cierre_dwn':'691466b7e0e4bd2690ca465e',
#     'cierre_base_efecto_Curva': '69171061d14aa54ed14d7305',
#     'cierre_up_efecto_Curva': '69171061d14aa54ed14d7305',
#     'cierre_base_efecto_balance': '69171986d14aa54ed155694f',
#     'cierre_up_efecto_balance': '69171c32d14aa54ed1597b09'
# }

load_ids = {
    'cierre_base': '69392ddb481cf43993eecdcd',
    'cierre_up':'693930bf481cf43993eecdce',
    'cierre_dwn':'6939317f481cf43993eecdcf',
    'cierre_base_efecto_curva': '69171061d14aa54ed14d7305',
    'cierre_up_efecto_curva': '69171061d14aa54ed14d7305',
    'cierre_base_efecto_balance': '69171986d14aa54ed155694f',
    'cierre_up_efecto_balance': '69171c32d14aa54ed1597b09'
}

### Query

In [3]:
fecha_sql_mes_anterior = '2025-10-01'

In [123]:
Informe_Efecto_Balance= f"""
        WITH metrics_agg AS (
        SELECT
            CASE
                WHEN dim_1 IN ('A_Cuentas en otras EECC', 'A_Tesoreria') THEN '1.1 BANCOS'
                WHEN dim_1 IN ('A_Ata') THEN '1.2 CARTERA MAYORISTA'
                WHEN dim_1 IN ('A_Depositos cedidos') THEN '1.3 DEPOSITOS OTRAS EECC'
                WHEN dim_1 IN (
                    'A_ECORP_Otros_Activo Corto Plazo',
                    'A_ECORP_Otros_Cred. Empresa_Gtia Aval',
                    'A_ECORP_Otros_Cred. Empresa_Gtia Personal',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Aval',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Personal',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Prenda',
                    'A_ECORP_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_ECORP_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_EESP_Otros_Activo Corto Plazo',
                    'A_EESP_Otros_Cred. Empresa_Gtia Aval',
                    'A_EESP_Otros_Cred. Empresa_Gtia Personal',
                    'A_EESP_Otros_Prest. Empresas_Gtia Aval',
                    /* ELIMINADO SEGÚN IMAGEN: 'A_EESP_Otros_Prest. Empresas_Gtia Personal', */
                    'A_EESP_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_EESP_Otros_Prest. Promotor_Gtia Hipot.',
                    'A_EESP_Otros_Prest. Promotor_Gtia Personal', /* AÑADIDO SEGÚN IMAGEN */
                    'A_PIB_Otros_Descubiertos_Gtia Personal',
                    'A_PIB_Otros_Prest. Consumo_Gtia Personal',
                    'A_PIB_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_PIB_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_PIB_Otros_Tarjetas Credito_Gtia Personal',
                    'A_POFI_Otros_Ant. Nomina_Gtia Personal',
                    'A_POFI_Otros_Descubiertos_Gtia Personal',
                    'A_POFI_Otros_Prest. Consumo_Gtia Personal',
                    'A_POFI_Otros_Prest. Consumo_Gtia Prenda',
                    'A_POFI_Otros_Prest. Empresas_Gtia Aval',
                    'A_POFI_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_POFI_Otros_Prest. Empresas_Gtia Personal',
                    'A_POFI_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_POFI_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_POFI_Otros_Prest. Origen_Gtia Aval',
                    'A_POFI_Otros_Prest. Origen_Gtia Hipot.',
                    'A_POFI_Otros_Tarjetas Credito_Gtia Personal',
                    'A_SSCC_BKIA_Prest. Consumo_Gtia Personal',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Aval',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Hipot.',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Personal',
                    'A_SSCC_Otros_Descubiertos_Gtia Personal',
                    'A_SSCC_Otros_Prest. Consumo_Gtia Personal',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Aval',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Personal',
                    'A_SSCC_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_SSCC_Otros_Tarjetas Credito_Gtia Personal',
                    'A_SSCCRSG_Otros_Tarjetas Credito_Gtia Personal'
                ) THEN '1.4 INVERSION CREDITICIA'
                WHEN dim_1 IN ('A_Bonos corporativos', 'A_Bonos soberanos') THEN 'RENTA FIJA'
                WHEN dim_1 = 'IRS_pago'      THEN 'IRS_pago'
                WHEN dim_1 = 'IRS_recibo'    THEN 'IRS_recibo'
                WHEN dim_1 = 'FXSWAP_pago'   THEN 'FXSWAP_pago'
                WHEN dim_1 = 'FXSWAP_recibo' THEN 'FXSWAP_recibo'
                WHEN dim_1 IN ('A_Cred._Dudoso', 'A_Efectos_Dudosos', 'A_Prest._Dudoso') THEN '1.6 ACTIVOS DUDOSOS'
                WHEN dim_1 IN ('A_Resto no sensible', 'A_Tesoreria_Admin.') THEN '1.7 ACTIVOS NO SENSIBLE'
                WHEN dim_1 = 'P_Cuentas de otras EECC' THEN '2.1 DEPOSITOS OTRAS EECC'
                WHEN dim_1 IN ('P_ECORP_P. Plazo', 'P_PIB_P. Plazo', 'P_POFI_P. Plazo', 'P_SSCC_P. Plazo') THEN '2.2 DEPOSITOS A PLAZO DE CLIENTES'
                WHEN dim_1 IN (
                    'P_ECORP_Ctas No Remuneradas', 'P_EESP_Ctas No Remuneradas', 'P_PIB_Ctas No Remuneradas',
                    'P_POFI_Ctas No Remuneradas', 'P_SSCC_Ctas No Remuneradas', 'P_SSCCFRO_Ctas No Remuneradas',
                    'P_PIB_Ctas Remuneradas', 'P_POFI_Ctas Remuneradas', 'P_SSCC_Ctas Remuneradas'
                ) THEN '2.3 DEPOSITOS A LA VISTA DE CLIENTES'
                WHEN dim_1 = 'P_Resto no sensible' THEN '2.4 PASIVOS NO SENSIBLES'
                WHEN dim_1 IN ('P_Ata', 'P_Emisiones T2') THEN '2.5 CARTERA MAYORISTA'
                ELSE 'OTROS'
            END AS Jerarquia,
            COALESCE(ROUND(SUM(CASE WHEN scenario = 'Base'
                    AND load_id = '{load_ids['cierre_base_efecto_balance']}'
                    THEN market_value ELSE 0 END) / 1000000, 2), 0) AS Base,
            COALESCE(ROUND(SUM(CASE WHEN scenario = 'Up200'
                    AND load_id = '{load_ids['cierre_up_efecto_balance']}'
                    THEN market_value ELSE 0 END) / 1000000, 2), 0) AS Up200
        FROM pro_pichincha_alquid_old.metric
        WHERE load_id IN (
            '{load_ids['cierre_base_efecto_balance']}',
            '{load_ids['cierre_up_efecto_balance']}'
        )
        GROUP BY 1
    ),

    jerarquias AS (
        SELECT '1.1 BANCOS'                      AS Jerarquia UNION ALL
        SELECT '1.2 CARTERA MAYORISTA'                      UNION ALL
        SELECT '1.3 DEPOSITOS OTRAS EECC'                   UNION ALL
        SELECT '1.4 INVERSION CREDITICIA'                   UNION ALL
        SELECT 'RENTA FIJA'                                 UNION ALL
        SELECT 'FXSWAP_pago'                                UNION ALL
        SELECT 'FXSWAP_recibo'                              UNION ALL
        SELECT 'IRS_pago'                                   UNION ALL
        SELECT 'IRS_recibo'                                 UNION ALL
        SELECT '1.6 ACTIVOS DUDOSOS'                        UNION ALL
        SELECT '1.7 ACTIVOS NO SENSIBLE'                    UNION ALL
        SELECT '2.1 DEPOSITOS OTRAS EECC'                   UNION ALL
        SELECT '2.2 DEPOSITOS A PLAZO DE CLIENTES'          UNION ALL
        SELECT '2.3 DEPOSITOS A LA VISTA DE CLIENTES'       UNION ALL
        SELECT '2.4 PASIVOS NO SENSIBLES'                   UNION ALL
        SELECT '2.5 CARTERA MAYORISTA'
    )

    , total_renta_fija AS (
        SELECT
            COALESCE(SUM(Base), 0)  AS Base,
            COALESCE(SUM(Up200), 0) AS Up200
        FROM metrics_agg
        WHERE Jerarquia IN (
            'RENTA FIJA',
            'FXSWAP_pago',
            'FXSWAP_recibo',
            'IRS_pago',
            'IRS_recibo'
        )
    )

    -- filas por jerarquía (si no hay saldo, salen Base/Up200 = NULL)
    SELECT
        j.Jerarquia,
        m.Base,
        m.Up200
    FROM jerarquias j
    LEFT JOIN metrics_agg m
    ON m.Jerarquia = j.Jerarquia

    UNION ALL
    -- TOTAL GRUPO 1: activos + derivados (igual que el filtro original)
        SELECT
            'TOTAL GRUPO 1' AS Jerarquia,
        (
            COALESCE(SUM(Base), 0)
            + (SELECT Base FROM total_renta_fija)
        ) AS Base,
        (
            COALESCE(SUM(Up200), 0)
            + (SELECT Up200 FROM total_renta_fija)
        ) AS Up200
    FROM metrics_agg
    WHERE Jerarquia IN (
        '1.1 BANCOS',
        '1.2 CARTERA MAYORISTA',
        '1.3 DEPOSITOS OTRAS EECC',
        '1.4 INVERSION CREDITICIA',
        '1.6 ACTIVOS DUDOSOS'
    )

    UNION ALL
    -- TOTAL GRUPO 2: pasivos (como en tu filtro: 2.1, 2.2, 2.3 y 2.5)
    SELECT
        'TOTAL GRUPO 2' AS Jerarquia,
        SUM(Base) AS Base,
        SUM(Up200) AS Up200
    FROM metrics_agg
    WHERE Jerarquia IN (
        '2.1 DEPOSITOS OTRAS EECC',
        '2.2 DEPOSITOS A PLAZO DE CLIENTES',
        '2.3 DEPOSITOS A LA VISTA DE CLIENTES',
        '2.5 CARTERA MAYORISTA'
    )

    UNION ALL
    -- TOTAL RENTA FIJA: bonos + derivados (como tu TOTAL RENTA FIJA original)
    SELECT
        'TOTAL RENTA FIJA' AS Jerarquia,
        SUM(Base) AS Base,
        SUM(Up200) AS Up200
    FROM metrics_agg
    WHERE Jerarquia IN (
        'RENTA FIJA',
        'FXSWAP_pago',
        'FXSWAP_recibo',
        'IRS_pago',
        'IRS_recibo'
    )

    ORDER BY CASE Jerarquia
        WHEN 'TOTAL GRUPO 1' THEN 1
        WHEN '1.1 BANCOS' THEN 2
        WHEN '1.2 CARTERA MAYORISTA' THEN 3
        WHEN '1.3 DEPOSITOS OTRAS EECC' THEN 4
        WHEN '1.4 INVERSION CREDITICIA' THEN 5
        WHEN 'TOTAL RENTA FIJA' THEN 6
        WHEN 'RENTA FIJA' THEN 7
        WHEN 'FXSWAP_pago' THEN 8
        WHEN 'FXSWAP_recibo' THEN 9
        WHEN 'IRS_pago' THEN 10
        WHEN 'IRS_recibo' THEN 11
        WHEN '1.6 ACTIVOS DUDOSOS' THEN 12
        WHEN '1.7 ACTIVOS NO SENSIBLE' THEN 13
        WHEN 'TOTAL GRUPO 2' THEN 14
        WHEN '2.5 CARTERA MAYORISTA' THEN 15
        WHEN '2.1 DEPOSITOS OTRAS EECC' THEN 16
        WHEN '2.2 DEPOSITOS A PLAZO DE CLIENTES' THEN 17
        WHEN '2.3 DEPOSITOS A LA VISTA DE CLIENTES' THEN 18
        WHEN '2.4 PASIVOS NO SENSIBLES' THEN 19
        ELSE 999
    END;
    """



In [124]:
df = pd.read_sql(Informe_Efecto_Balance, CONN)


C:\Users\EM2024007370\AppData\Local\Temp\ipykernel_936\4070727595.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(Informe_Efecto_Balance, CONN)


In [126]:
df


,Jerarquia,Base,Up200
0,TOTAL GRUPO 1,3933.98,3835.01
1,1.1 BANCOS,294.73,293.79
2,1.2 CARTERA MAYORISTA,NaN,NaN
3,1.3 DEPOSITOS OTRAS EECC,2.34,2.31
4,1.4 INVERSION CREDITICIA,2443.83,2409.80
5,TOTAL RENTA FIJA,1169.38,1106.21
6,RENTA FIJA,1143.88,1063.13
7,FXSWAP_pago,-18.94,-18.80
8,FXSWAP_recibo,19.44,19.30
9,IRS_pago,-82.78,-63.40


In [83]:
df["Up200"][1:6].sum()

np.float64(3812.11)

### Pruebas MF CTO

In [8]:
lista = ['A_ECORP_Otros_Activo Corto Plazo',
                    'A_ECORP_Otros_Cred. Empresa_Gtia Aval',
                    'A_ECORP_Otros_Cred. Empresa_Gtia Personal',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Aval',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Personal',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Prenda',
                    'A_ECORP_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_ECORP_Otros_Prest. Hipotecas_Gtia Personal',

                    'A_EESP_Otros_Activo Corto Plazo',
                    'A_EESP_Otros_Cred. Empresa_Gtia Aval',
                    'A_EESP_Otros_Cred. Empresa_Gtia Personal',
                    'A_EESP_Otros_Prest. Empresas_Gtia Aval',
                    'A_EESP_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_EESP_Otros_Prest. Promotor_Gtia Aval',
                    'A_EESP_Otros_Prest. Promotor_Gtia Hipot.',
                    'A_EESP_Otros_Prest. Promotor_Gtia Personal',

                    'A_PIB_Otros_Descubiertos_Gtia Personal',
                    'A_PIB_Otros_Prest. Consumo_Gtia Personal',
                    'A_PIB_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_PIB_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_PIB_Otros_Tarjetas Credito_Gtia Personal',

                    'A_POFI_Otros_Ant. Nomina_Gtia Personal',
                    'A_POFI_Otros_Descubiertos_Gtia Personal',
                    'A_POFI_Otros_Prest. Consumo_Gtia Personal',
                    'A_POFI_Otros_Prest. Consumo_Gtia Prenda',
                    'A_POFI_Otros_Prest. Empresas_Gtia Aval',
                    'A_POFI_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_POFI_Otros_Prest. Empresas_Gtia Personal',
                    'A_POFI_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_POFI_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_POFI_Otros_Prest. Origen_Gtia Aval',
                    'A_POFI_Otros_Prest. Origen_Gtia Hipot.',
                    'A_POFI_Otros_Tarjetas Credito_Gtia Personal',

                    'A_SSCC_BKIA_Prest. Consumo_Gtia Personal',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Aval',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Personal',
                    'A_SSCC_Otros_Descubiertos_Gtia Personal',
                    'A_SSCC_Otros_Prest. Consumo_Gtia Personal',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Aval',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Personal',
                    'A_SSCC_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_SSCC_Otros_Tarjetas Credito_Gtia Personal',

                    'A_SSCCRSG_Otros_Tarjetas Credito_Gtia Personal']

In [9]:
complete = True
x = 0

for i in original:
    if i not in lista:
        complete = False
        x +=1
        print(i)



x

NameError: name 'original' is not defined

### Pruebas VE Sensibilidades

In [ ]:
lista = ['A_Cuentas en otras EECC', 'A_Tesoreria', 'A_Ata', 'A_Depositos cedidos',
    'A_ECORP_Otros_Activo Corto Plazo', 'A_EESP_Otros_Activo Corto Plazo', 'A_POFI_Otros_Ant. Nomina_Gtia Personal',
    'A_ECORP_Otros_Cred. Empresa_Gtia Aval', 'A_ECORP_Otros_Cred. Empresa_Gtia Personal',
    'A_EESP_Otros_Cred. Empresa_Gtia Aval', 'A_EESP_Otros_Cred. Empresa_Gtia Personal',
    'A_SSCC_Otros_Cred. Empresa_Gtia Aval', 'A_SSCC_Otros_Cred. Empresa_Gtia Hipot.', 'A_SSCC_Otros_Cred. Empresa_Gtia Personal',
    'A_PIB_Otros_Descubiertos_Gtia Personal', 'A_POFI_Otros_Descubiertos_Gtia Personal', 'A_SSCC_Otros_Descubiertos_Gtia Personal',
    'A_ECORP_Otros_Prest. Empresas_Gtia Aval', 'A_ECORP_Otros_Prest. Empresas_Gtia Hipot.', 'A_ECORP_Otros_Prest. Empresas_Gtia Personal',
    'A_ECORP_Otros_Prest. Empresas_Gtia Prenda', 'A_ECORP_Otros_Prest. Hipotecas_Gtia Hipot.', 'A_ECORP_Otros_Prest. Hipotecas_Gtia Personal',
    'A_EESP_Otros_Prest. Empresas_Gtia Aval', 'A_EESP_Otros_Prest. Empresas_Gtia Personal', 'A_EESP_Otros_Prest. Hipotecas_Gtia Hipot.',
    'A_EESP_Otros_Prest. Promotor_Gtia Hipot.', 'A_PIB_Otros_Prest. Consumo_Gtia Personal', 'A_PIB_Otros_Prest. Hipotecas_Gtia Hipot.',
    'A_PIB_Otros_Prest. Hipotecas_Gtia Personal', 'A_POFI_Otros_Prest. Consumo_Gtia Personal', 'A_POFI_Otros_Prest. Consumo_Gtia Prenda',
    'A_POFI_Otros_Prest. Empresas_Gtia Aval', 'A_POFI_Otros_Prest. Empresas_Gtia Hipot.', 'A_POFI_Otros_Prest. Empresas_Gtia Personal',
    'A_POFI_Otros_Prest. Hipotecas_Gtia Hipot.', 'A_POFI_Otros_Prest. Hipotecas_Gtia Personal',
    'A_POFI_Otros_Prest. Origen_Gtia Aval', 'A_POFI_Otros_Prest. Origen_Gtia Hipot.',
    'A_SSCC_BKIA_Prest. Consumo_Gtia Personal', 'A_SSCC_Otros_Prest. Consumo_Gtia Personal',
    'A_SSCC_Otros_Prest. Empresas_Gtia Aval', 'A_SSCC_Otros_Prest. Empresas_Gtia Hipot.',
    'A_SSCC_Otros_Prest. Empresas_Gtia Personal', 'A_SSCC_Otros_Prest. Hipotecas_Gtia Hipot.',
    'A_PIB_Otros_Tarjetas Credito_Gtia Personal', 'A_POFI_Otros_Tarjetas Credito_Gtia Personal',
    'A_SSCC_Otros_Tarjetas Credito_Gtia Personal', 'A_SSCCRSG_Otros_Tarjetas Credito_Gtia Personal',
    'A_Bonos corporativos', 'A_Bonos soberanos', 'A_Cred._Dudoso', 'A_Efectos_Dudosos', 'A_Prest._Dudoso',
    'A_Resto no sensible', 'A_Tesoreria_Admin.']

In [ ]:
complete = True
x = 0

for i in original:
    if i not in lista:
        complete = False
        x +=1
        print(i)



x

A_EESP_Otros_Prest. Promotor_Gtia Aval
A_EESP_Otros_Prest. Promotor_Gtia Personal


2

In [ ]:
len(lista)

56

### Pruebas MF

#### Chequeo que usan los mismos campos de inversión crediticia

In [ ]:
code = [
                        'A_ECORP_Otros_Activo Corto Plazo',
                    'A_ECORP_Otros_Cred. Empresa_Gtia Aval',
                    'A_ECORP_Otros_Cred. Empresa_Gtia Personal',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Aval',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Personal',
                    'A_ECORP_Otros_Prest. Empresas_Gtia Prenda',
                    'A_ECORP_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_ECORP_Otros_Prest. Hipotecas_Gtia Personal',

                    'A_EESP_Otros_Activo Corto Plazo',
                    'A_EESP_Otros_Cred. Empresa_Gtia Aval',
                    'A_EESP_Otros_Cred. Empresa_Gtia Personal',
                    'A_EESP_Otros_Prest. Empresas_Gtia Aval',
                    'A_EESP_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_EESP_Otros_Prest. Promotor_Gtia Aval',
                    'A_EESP_Otros_Prest. Promotor_Gtia Hipot.',
                    'A_EESP_Otros_Prest. Promotor_Gtia Personal',

                    'A_PIB_Otros_Descubiertos_Gtia Personal',
                    'A_PIB_Otros_Prest. Consumo_Gtia Personal',
                    'A_PIB_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_PIB_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_PIB_Otros_Tarjetas Credito_Gtia Personal',

                    'A_POFI_Otros_Ant. Nomina_Gtia Personal',
                    'A_POFI_Otros_Descubiertos_Gtia Personal',
                    'A_POFI_Otros_Prest. Consumo_Gtia Personal',
                    'A_POFI_Otros_Prest. Consumo_Gtia Prenda',
                    'A_POFI_Otros_Prest. Empresas_Gtia Aval',
                    'A_POFI_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_POFI_Otros_Prest. Empresas_Gtia Personal',
                    'A_POFI_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_POFI_Otros_Prest. Hipotecas_Gtia Personal',
                    'A_POFI_Otros_Prest. Origen_Gtia Aval',
                    'A_POFI_Otros_Prest. Origen_Gtia Hipot.',
                    'A_POFI_Otros_Tarjetas Credito_Gtia Personal',

                    'A_SSCC_BKIA_Prest. Consumo_Gtia Personal',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Aval',
                    'A_SSCC_Otros_Cred. Empresa_Gtia Personal',
                    'A_SSCC_Otros_Descubiertos_Gtia Personal',
                    'A_SSCC_Otros_Prest. Consumo_Gtia Personal',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Aval',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Hipot.',
                    'A_SSCC_Otros_Prest. Empresas_Gtia Personal',
                    'A_SSCC_Otros_Prest. Hipotecas_Gtia Hipot.',
                    'A_SSCC_Otros_Tarjetas Credito_Gtia Personal',

                    'A_SSCCRSG_Otros_Tarjetas Credito_Gtia Personal'
]

In [ ]:
original = ["A_ECORP_Otros_Activo Corto Plazo",
"A_EESP_Otros_Activo Corto Plazo",
"A_POFI_Otros_Ant. Nomina_Gtia Personal",
"A_ECORP_Otros_Cred. Empresa_Gtia Aval",
"A_ECORP_Otros_Cred. Empresa_Gtia Personal",
"A_EESP_Otros_Cred. Empresa_Gtia Aval",
"A_EESP_Otros_Cred. Empresa_Gtia Personal",
"A_SSCC_Otros_Cred. Empresa_Gtia Aval",
"A_SSCC_Otros_Cred. Empresa_Gtia Personal",
"A_PIB_Otros_Descubiertos_Gtia Personal",
"A_POFI_Otros_Descubiertos_Gtia Personal",
"A_SSCC_Otros_Descubiertos_Gtia Personal",
"A_ECORP_Otros_Prest. Empresas_Gtia Aval",
"A_ECORP_Otros_Prest. Empresas_Gtia Hipot.",
"A_ECORP_Otros_Prest. Empresas_Gtia Personal",
"A_ECORP_Otros_Prest. Empresas_Gtia Prenda",
"A_ECORP_Otros_Prest. Hipotecas_Gtia Hipot.",
"A_ECORP_Otros_Prest. Hipotecas_Gtia Personal",
"A_EESP_Otros_Prest. Empresas_Gtia Aval",
"A_EESP_Otros_Prest. Hipotecas_Gtia Hipot.",
"A_EESP_Otros_Prest. Promotor_Gtia Aval",
"A_EESP_Otros_Prest. Promotor_Gtia Hipot.",
"A_EESP_Otros_Prest. Promotor_Gtia Personal",
"A_PIB_Otros_Prest. Consumo_Gtia Personal",
"A_PIB_Otros_Prest. Hipotecas_Gtia Hipot.",
"A_PIB_Otros_Prest. Hipotecas_Gtia Personal",
"A_POFI_Otros_Prest. Consumo_Gtia Personal",
"A_POFI_Otros_Prest. Consumo_Gtia Prenda",
"A_POFI_Otros_Prest. Empresas_Gtia Aval",
"A_POFI_Otros_Prest. Empresas_Gtia Hipot.",
"A_POFI_Otros_Prest. Empresas_Gtia Personal",
"A_POFI_Otros_Prest. Hipotecas_Gtia Hipot.",
"A_POFI_Otros_Prest. Hipotecas_Gtia Personal",
"A_POFI_Otros_Prest. Origen_Gtia Aval",
"A_POFI_Otros_Prest. Origen_Gtia Hipot.",
"A_SSCC_BKIA_Prest. Consumo_Gtia Personal",
"A_SSCC_Otros_Prest. Consumo_Gtia Personal",
"A_SSCC_Otros_Prest. Empresas_Gtia Aval",
"A_SSCC_Otros_Prest. Empresas_Gtia Hipot.",
"A_SSCC_Otros_Prest. Empresas_Gtia Personal",
"A_SSCC_Otros_Prest. Hipotecas_Gtia Hipot.",
"A_PIB_Otros_Tarjetas Credito_Gtia Personal",
"A_POFI_Otros_Tarjetas Credito_Gtia Personal",
"A_SSCC_Otros_Tarjetas Credito_Gtia Personal",
"A_SSCCRSG_Otros_Tarjetas Credito_Gtia Personal"]


In [ ]:
len(code)

45

In [ ]:
complete = True

for i in original:
    if i not in code:
        complete = False


complete


True

In [ ]:
complete = True

for i in code:
    if i not in original:
        complete = False


complete

True

#### Chequear valores de las matcheadas por el patrón LIKE para CREDITOS

In [ ]:
query = f"""
SELECT DISTINCT dim_1
FROM pro_pichincha_alquid_old."result"
WHERE dim_1 LIKE '%_Cred.%'
AND load_id IN (
        '{load_ids['cierre_base']}',
        '{load_ids['cierre_up']}',
        '{load_ids['cierre_dwn']}'
        )
ORDER BY dim_1;
"""

df = pd.read_sql(query, CONN)
df

C:\Users\EM2024007370\AppData\Local\Temp\ipykernel_24260\1613284560.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, CONN)


,dim_1
0,A_Cred._Dudoso
1,A_ECORP_Otros_Cred. Empresa_Gtia Aval
2,A_ECORP_Otros_Cred. Empresa_Gtia Personal
3,A_EESP_Otros_Cred. Empresa_Gtia Aval
4,A_EESP_Otros_Cred. Empresa_Gtia Personal
5,A_SSCC_Otros_Cred. Empresa_Gtia Aval
6,A_SSCC_Otros_Cred. Empresa_Gtia Personal
